In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
import math

### Goal of the project
In Asian Options, the pricing strategy is as follows : <br>
The value of the call option at expiry is given by : $\max(\text{avg}(S) - K,0)$ 
where we take the average of the stock in a given time interval. So we need to find out the value of the call option. We do it using Binomial Tree Method.

### Binomial Tree Model. 
For each level, we have the following data :
1) Price of the stock $S_{i,j}$ where $i$ denotes the level.
2) Average price of the stock $A_{i,j}$ --> Justify why there is a need to compute $A_{i,j}$ at every state!
We are pricing via the paths. Suppose there are 5 levels. There are 2^5 path possibilites ( either up/down in 5 places _,_,_,_,_ ).  For each of the path, we calculate the price of the stock and the average price. 
3) So the price $S_{i,j}$ and the average price $A_{i,j}$ are saved in a node.

Hence these are best represented as a class. In a node, we have the following details : Stock price, and the average price of all the paths whose end point is the stock.

In [6]:
class Node:
    """ A class for storing the price and average at a node  """
    def __init__(self, S: float, avg: float): # For initialization of values
        self.S=S        # current price
        self.avg=avg      # average so far
    def __repr__(self): # For printing the price --> When I used str it didn't work
        return f"Node(S={self.S:.2f}, avg={self.avg:.2f})"


### Explanation for Binomial Tree Model. 
We will now describe the Binomial Tree Model for Asian options and will focus on Arithmetic Average. 
1) The Payoff depends on the average of prices along the path i.e. Payoff = $\max (\overline{S} - K,0)$ where 
$\overline{S}$ denotes the average of the prices till time N
2) So, we now have a tree model for the stock prices ( at each level the number of entries increases by 1 ). This is because $S_{i,j+1}$ is either $S_{i,j} u$ or $S_{i,j}d$
3) Next, for each level we compute the average price. This can be attained by a recursion relation :
   $$A_{i+1,j} = \frac{i A_{i,j} + uS_{i,j}}{i+1} \qquad \text{( in case the stock goes up )} $$
   $$A_{i+1,j+1} = \frac{i A_{i,j} + dS_{i,j}}{i+1} \qquad \text {( in case the stock goes down ) }$$
4) At the time of maturity for each of the $2^N$ possibilities, we compute the payoff which given by $V_{N,j} = \max(A_{i,j} - K,0)$. ( It is perhaps a poor choice of notation here )
5) Finally to compute the price of the option for $i<N$, we use the following relation <br>
    $V_{i,j} = e^{-rt} E[ V_{i+1,j} ] = V_{i,j} = e^{-rt} p V_{i,j} + (1-p) V_{i,j+1} $ <br>
    Here r is the risk free rate, p is the risk neutral probability given by $p = (e^{r \Delta t} - d)/(u-d)$ <br>

#### Modification for computation of option price at time t=0
We will elaborate on the last point and simplify the demonstration for computing $V_{0,0}$. Since the discounted option prices are martingales we can write <br> 
1) $ V_{0,0} = E[e^{-rT} V_{N}] = e^{-r} E[V_N] $ <br>
2) We have the risk netural probability $p$ and at step $N$ the Expectation can be expanded as follows : <br> 
   $E[V_N]= \sum_{j=0}^N p^{N-j}(1-p)^{j} V_{N,j}.$<br> 
   Here $V_{N,j}$ is the sum of values of the option prices $tree[N][j]$. Recall that $tree[N][j]$ has $\binom{N}{j}$ entries and each entry occurs with probability $p^{N-j} (1-p)^{j}$. <br>
3) However it seems a bit expensive to compute the coefficients of $V_{N,j}$ (more precisely to compute powers $p^j (1-p)^{N-j}$ repeatedly; so we use a dynamic programming approach<br>
   At 0, the coefficient is $(1-p)^n$ so let this be the first entry of the list L. <br>
   Everytime we append $L[-1]*(p/1-p)$ where k ranges from 0 to $N-1$ <br> 
4) After creating this list L, we can simply multiply the elements individually and take their sum( which is saved at the list <bf>{Option_value_at_expiry}<bf>

In [4]:

def payoff_binomial_tree_asian_option(S0, u, d, r, N, K):   
    """ Uses the binomial tree model to return the price of an Asian call option at time t=0""" 
    tree = [[[] for j in range(i + 1)] for i in range(N + 1)]
    tree[0][0].append(Node(S0, S0))  # Step 0, initial node
    for i in range(N):
        for j in range(i + 1):
            for node in tree[i][j]:
                # Up move
                Su = node.S * u
                Au = (node.avg * i + Su) / (i + 1)
                tree[i + 1][j + 1].append(Node(Su, Au))
                # Down move
                Sd = node.S * d
                Ad = (node.avg * i + Sd) / (i + 1)
                tree[i + 1][j].append(Node(Sd, Ad))
    # Completed the tree construction with average prices at each node.
    Option_Val_at_expiry=[];
    for j in range(N+1):
        expiry_value=sum([max(node.avg - K, 0) for node in tree[N][j]]);
        Option_Val_at_expiry.append(expiry_value);
    dt = 1/N
    p = (np.exp(r*dt) - d) / (u - d)  # risk-neutral probability
    Coefficients=[(1-p)**N]
    for j in range(N):
        Coefficients.append(Coefficients[-1]*p/((1-p)))
    Option_Val_at_start=sum([Coefficients[i]*Option_Val_at_expiry[i] for i in range(0,N+1)])*np.exp(-r);
    Option_Val_at_start=np.exp(-r)*Option_Val_at_start;
    print(Option_Val_at_start);
    return(Option_Val_at_start);

In [5]:
# Demonstration of the pricing model: The initial setup 5 period binomial model.
S0=100
u=1.1
d=0.9
r=0.05
N=5;
K = 110;
payoff_binomial_tree_asian_option(S0, u, d, r, N, K);

NameError: name 'np' is not defined